### University of Michigan Survey of Consumers

In [1]:
import sys
sys.path.append('../src')
import uschartbook.config
from uschartbook.config import *
from uschartbook.utils import *

In [2]:
url = 'http://www.sca.isr.umich.edu/files/tbmics.csv'

df = pd.read_csv(url)
df.index = pd.to_datetime(df['Month'] + ' ' + df['YYYY'].astype('str'), format='%B %Y')
data = df['ICS_ALL'].loc['1989':]

data.to_csv(data_dir / 'umichsoc.csv', index_label='date', header='True')

In [3]:
data = pd.read_csv(data_dir / 'umichsoc.csv').set_index('date')['ICS_ALL']
data.index = pd.to_datetime(data.index)
write_txt(text_dir / 'soc_node.txt', end_node(data, 'violet!60!blue', date='m'))

d = {}
for i in [-1, -2, -13]:
    d[i] = {'date': dtxt(data.index[i])['mon1'], 'val': data.iloc[i]}
pcv = data.loc['2019-03-01':'2020-02-01'].mean()
ch_val = -(1 - (d[-1]['val'] / pcv)) * 100
ch_txt = value_text(ch_val, style='above_below', threshold=0.5)
text = (f'As of {d[-1]["date"]}, the latest value of the consumer sentiment index is '+
        f'{d[-1]["val"]}, following {d[-2]["val"]} in {d[-2]["date"]}, and compared to '+
        f'{d[-13]["val"]} one year prior, in {d[-13]["date"]}. '+
        f'As a pre-COVID baseline, the index average value was {pcv:.1f} '+
        'during the year ending February 2020; the consumer '+
        f'sentiment index is currently {ch_txt} this level.')
write_txt(text_dir / 'umichsoc.txt', text)
print(text)

As of February 2026, the latest value of the consumer sentiment index is 56.6, following 56.4 in January 2026, and compared to 64.7 one year prior, in February 2025. As a pre-COVID baseline, the index average value was 97.3 during the year ending February 2020; the consumer sentiment index is currently 41.8 percent below this level.


### Current conditions and expections

In [4]:
url = 'http://www.sca.isr.umich.edu/files/tbmiccice.csv'
df = pd.read_csv(url)
df.index = pd.to_datetime(df['Month'] + ' ' + df['YYYY'].astype('str'), format='%B %Y')
df = df.loc['1989':, ['ICC', 'ICE']]
df.to_csv(data_dir / 'umichsoc_ice.csv', index_label='date')

In [5]:
df = pd.read_csv(data_dir / 'umichsoc_ice.csv', index_col='date', 
                 parse_dates=True)
ltdt = dtxt(df.index[-1])['mon1']
prdt = dtxt(df.index[-2])['mon1']
ccval = df.ICC.iloc[-1]
ccpr = df.ICC.iloc[-2]
cc19 = df.loc['2019', 'ICC'].mean()
cccol = 'skyblue'
ccnode = end_node(df.ICC, cccol, date='m')
write_txt(text_dir / 'icc_node.txt', ccnode)

ceval = df.ICE.iloc[-1]
cepr = df.ICE.iloc[-2]
ce19 = df.loc['2019', 'ICE'].mean()
cecol = 'forestgreen'
text = ('The consumer sentiment index combines views on current '+
        f'and future economic conditions. In {ltdt}, the index tracking '+
        f'views on current economic conditions was {ccval:.1f}, '+
        f'compared to {ccpr:.1f} in {prdt}, and {cc19:.1f} '+
        f'in 2019 {c_line(cccol)}.\n\n'+
        f'In {ltdt}, the index tracking consumer '+
        f'expectations for future economic conditions was {ceval:.1f}, '+
        f'compared to {cepr:.1f} in {prdt}, and {ce19:.1f} '+
        f'in 2019 {c_line(cecol)}.')
write_txt(text_dir / 'umichsoc_ice.txt', text)
print(text)
ccnode = end_node(df.ICC, cccol, date='m')
write_txt(text_dir / 'icc_node.txt', ccnode)
cenode = end_node(df.ICE, cecol, date='m')
write_txt(text_dir / 'ice_node.txt', cenode)

The consumer sentiment index combines views on current and future economic conditions. In February 2026, the index tracking views on current economic conditions was 56.6, compared to 55.4 in January 2026, and 110.8 in 2019 (see {\color{skyblue}\textbf{---}}).

In February 2026, the index tracking consumer expectations for future economic conditions was 56.6, compared to 57.0 in January 2026, and 86.5 in 2019 (see {\color{forestgreen}\textbf{---}}).


### Inflation Expecations (One and Five Years Ahead)

In [6]:
url = 'http://www.sca.isr.umich.edu/files/tbmpx1px5.csv'
dfm = pd.read_csv(url)
dfm.index = pd.to_datetime(dfm['Month'] + ' ' + dfm['YYYY'].astype('str'), format='%B %Y')
dfm = dfm.drop(['Month', 'YYYY'], axis=1)

url = 'http://www.sca.isr.umich.edu/files/tbcpx1px5.csv'
df = pd.read_csv(url, skiprows=3).dropna(axis=1, how='all').dropna()
df['DATE OF SURVEY'] = df['DATE OF SURVEY'].str.replace(' (P)', '', regex=False)
df.columns = ['Month', 'YYYY', 'PX_MD', 'PX5_MD']
df.index = pd.to_datetime(df['Month'] + ' ' + df['YYYY'].astype('int').astype('str'), format='%B %Y')
df = df.drop(['Month', 'YYYY'], axis=1)

if df.index[-1] not in dfm.index:
    dfm = pd.concat([dfm, df.iloc[-1].to_frame().T])
    
dfm.resample('QS').mean().loc['1990':].round(4).to_csv(data_dir / 'infumichlt.csv', index_label='date')    
data = dfm['PX5_MD'].loc['2018':]
data.to_csv(data_dir / 'infumich.csv', index_label='date', header='VALUE')
color = 'violet!60!magenta'
node = end_node(data, color)
write_txt(text_dir / 'infumich_node.txt', node)  

ltdt = dtxt(data.index[-1])['mon1']
prdt = dtxt(data.index[-13])['mon1']
p5val = data.iloc[-61]
lval = data.iloc[-1]
pval = data.iloc[-13]

inf_act = pd.read_csv(data_dir / 'cpi.csv')['ALL'].iloc[-60:].mean()
text = (f'As of {ltdt}, surveyed consumers expect inflation to average '+
        f'{lval} percent over the next five years {c_line(color)}, '+
        f'compared to an expected rate of {pval} percent in {prdt}. '+
        f'Consumers had expected inflation to average {p5val} percent over '+
        f'the past five years, while actual inflation over the period '+
        f'was {inf_act:.1f} percent.')
write_txt(text_dir / 'inf_exp_cons.txt', text)
print(text)

yrval = dfm['PX_MD'].iloc[-1]
cl = '(see \\tikz[baseline=-1mm]\\draw[black!88!white,ultra thick,densely dashed](0,0)--+(0.33,0);)'
text = (f'Respondents expect consumer prices to increase {yrval} '+
        f'percent over the year starting {ltdt} {cl}.')
write_txt(text_dir / 'inf_exp_surv_st.txt', text)
print(text)

As of March 2026, surveyed consumers expect inflation to average 3.2 percent over the next five years (see {\color{violet!60!magenta}\textbf{---}}), compared to an expected rate of 4.1 percent in March 2025. Consumers had expected inflation to average 2.8 percent over the past five years, while actual inflation over the period was 4.5 percent.
Respondents expect consumer prices to increase 3.4 percent over the year starting March 2026 (see \tikz[baseline=-1mm]\draw[black!88!white,ultra thick,densely dashed](0,0)--+(0.33,0);).
